In [0]:
import requests
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

YEAR = 2025
sessions_url = f"https://api.openf1.org/v1/sessions?year={YEAR}&session_type=Race"

def ingest_2025_safe():
    response = requests.get(sessions_url)
    if response.status_code == 200:
        session_data = response.json()
        
        # FIX: Convert to Pandas first to handle null types automatically
        pdf_sessions = pd.DataFrame(session_data)
        sessions_df = spark.createDataFrame(pdf_sessions)
        
        sessions_df.withColumn("ingested_at", current_timestamp()) \
            .write.format("delta").mode("overwrite") \
            .saveAsTable("f1_project.bronze.raw_sessions_2025")
        
        latest_session_key = session_data[-1]['session_key']
        print(f"Latest 2025 Race Session Key: {latest_session_key}")

        race_endpoints = {
            "raw_drivers_2025": f"https://api.openf1.org/v1/drivers?session_key={latest_session_key}",
            "raw_laps_2025": f"https://api.openf1.org/v1/laps?session_key={latest_session_key}",
            "raw_telemetry_2025": f"https://api.openf1.org/v1/car_data?session_key={latest_session_key}&speed>250"
        }

        for table_name, url in race_endpoints.items():
            res = requests.get(url)
            if res.status_code == 200:
                data = res.json()
                if data:
                    # FIX: Using Pandas as a middle-man to prevent CANNOT_DETERMINE_TYPE
                    pdf = pd.DataFrame(data)
                    
                    # Convert all objects to strings to ensure Spark doesn't trip on mixed types
                    pdf = pdf.astype(str) 
                    
                    df = spark.createDataFrame(pdf)
                    df.withColumn("ingested_at", current_timestamp()) \
                      .write.format("delta").mode("overwrite") \
                      .saveAsTable(f"f1_project.bronze.{table_name}")
                    print(f"✅ Ingested 2025 {table_name}")
                    
    else:
        print(f"Error: {response.status_code}")

ingest_2025_safe()